In [ ]:
# ============================================================
# DeepSVDD working on CNN Latent Space
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- 1. CNN Encoder (you can reuse the one already in 03) ----------
class CNNEncoder1D(nn.Module):
    def __init__(self, n_features=5, seq_len=30, embed_dim=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Linear(64, embed_dim)

    def forward(self, x):
        # x: [batch, seq_len, n_features]
        x = x.permute(0, 2, 1)
        x = self.conv(x).squeeze(-1)
        return self.fc(x)


# ---------- 2. DeepSVDD that works on CNN embeddings ----------
class DeepSVDD_CNN:
    def __init__(self, cnn_encoder, rep_dim=32, lr=1e-3):
        self.cnn = cnn_encoder.to(device)
        self.rep_dim = rep_dim
        self.c = None  # hypersphere center

        # We only train the CNN (or freeze it later)
        self.optimizer = optim.Adam(self.cnn.parameters(), lr=lr, weight_decay=1e-6)

    def init_center(self, dataloader):
        """Compute center c from normal data embeddings"""
        self.cnn.eval()
        embeddings = []

        with torch.no_grad():
            for batch in dataloader:
                # batch can be (tabular, sequence) or just sequence
                if isinstance(batch, (list, tuple)):
                    seq = batch[-1] if len(batch) > 1 else batch[0]
                else:
                    seq = batch
                seq = seq.to(device).float()
                z = self.cnn(seq)
                embeddings.append(z)

        embeddings = torch.cat(embeddings, dim=0)
        self.c = embeddings.mean(dim=0)
        print(f"Center c initialized → shape {self.c.shape}")

    def train(self, dataloader, epochs=15):
        if self.c is None:
            self.init_center(dataloader)

        self.cnn.train()
        for epoch in range(epochs):
            total_loss = 0.0
            for batch in dataloader:
                if isinstance(batch, (list, tuple)):
                    seq = batch[-1] if len(batch) > 1 else batch[0]
                else:
                    seq = batch
                seq = seq.to(device).float()

                self.optimizer.zero_grad()
                z = self.cnn(seq)
                dist = torch.sum((z - self.c) ** 2, dim=1)
                loss = dist.mean()
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()

            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{epochs} | Avg Distance: {total_loss/len(dataloader):.6f}")

        print("✅ DeepSVDD + CNN training completed.")

    def anomaly_score(self, sequence):
        """Higher score = more anomalous"""
        self.cnn.eval()
        with torch.no_grad():
            sequence = sequence.to(device).float()
            z = self.cnn(sequence)
            scores = torch.sum((z - self.c) ** 2, dim=1)
        return scores.cpu().numpy()

    def predict(self, sequence, threshold=None):
        scores = self.anomaly_score(sequence)
        if threshold is None:
            threshold = np.percentile(scores, 95)
        preds = (scores > threshold).astype(int)
        return preds, scores, threshold